In [2]:
import google.generativeai as genai
import os
import re

def get_api_key(filepath="apikey.txt"):
    """指定されたファイルからAPIキーを読み込む"""
    try:
        with open(filepath, "r") as f:
            return f.read().strip()
    except FileNotFoundError:
        print(f"エラー: APIキーファイル '{filepath}' が見つかりません。")
        return None

def solve_historical_ordering_few_shot(api_key, examples, new_problem_text):
    """
    Gemini APIを使用して、Few-Shot推論で歴史的な出来事を年代順に並べ替える。

    Args:
        api_key (str): Gemini APIキー。
        examples (list): (問題文, 解答文字列) のタプルのリスト。
        new_problem_text (str): 解答を生成すべき新しい問題文。

    Returns:
        str: 年代順に並べられた出来事の記号の文字列 (例: "イ→ウ→ア")。
             エラーが発生した場合はNone。
    """
    if not api_key:
        return None

    genai.configure(api_key=api_key)
    # model = genai.GenerativeModel('gemini-2.0-flash-lite')
    model = genai.GenerativeModel('gemini-2.0-flash-lite') # Few-shotにはより高性能なモデルが適している場合がある

    # プロンプトの構築
    prompt_parts = ["以下の問題と解答の例を参考に、最後の問題に解答してください。解答は「記号→記号→記号」の形式で答えてください。\n"]

    for i, (problem, solution) in enumerate(examples):
        prompt_parts.append(f"例{i+1}:")
        prompt_parts.append(f"問題:\n{problem.strip()}")
        prompt_parts.append(f"解答: {solution.strip()}\n")

    prompt_parts.append("問題:")
    prompt_parts.append(new_problem_text.strip())
    prompt_parts.append("解答:") # モデルにこの後に続けて解答を生成させる

    full_prompt = "\n".join(prompt_parts)

    # print("---送信するプロンプト---")
    # print(full_prompt)
    # print("----------------------")

    try:
        print("\nGeminiに問い合わせ中 (Few-Shot推論)...")
        response = model.generate_content(full_prompt)
        # print(f"---APIからの生レスポンス---\n{response.text}\n--------------------------")

        # 解答部分の抽出 (例: "ア→イ→ウ" の形式を期待)
        # モデルが "解答: ア→イ→ウ" のように返すこともあれば、"ア→イ→ウ" のみを返すこともある
        # より堅牢にするために、応答テキスト全体からパターンマッチングする
        answer_text = response.text.strip()

        # "解答: X→Y→Z" または "X→Y→Z" のパターンを探す
        match = re.search(r"([ア-ンｱ-ﾝｧ-ｮァ-ョ])→([ア-ンｱ-ﾝｧ-ｮァ-ョ])→([ア-ンｱ-ﾝｧ-ｮァ-ョ])", answer_text)
        if match:
            extracted_answer = match.group(0)
            print(f"  抽出された解答: {extracted_answer}")
            return extracted_answer
        else:
            # もし上記のパターンで見つからなければ、レスポンスの最後の非空白行を試す
            lines = [line for line in answer_text.splitlines() if line.strip()]
            if lines:
                last_line = lines[-1]
                # 再度パターンマッチを試みる
                match_last_line = re.search(r"([ア-ンｱ-ﾝｧ-ｮァ-ョ])→([ア-ンｱ-ﾝｧ-ｮァ-ョ])→([ア-ンｱ-ﾝｧ-ｮァ-ョ])", last_line)
                if match_last_line:
                    extracted_answer = match_last_line.group(0)
                    print(f"  抽出された解答 (最終行から): {extracted_answer}")
                    return extracted_answer
            print(f"  エラー: 解答の形式が不正です。レスポンス: {answer_text}")
            return None

    except Exception as e:
        print(f"  API呼び出しまたはレスポンス処理中にエラーが発生しました: {e}")
        return None

if __name__ == "__main__":
    # Few-Shot学習のための例題と解答
    examples = [
        (
            """日本の近代化に関連するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　府知事・県令からなる地方官会議が設置された。
イ　廃藩置県が実施され，中央から府知事・県令が派遣される体制になった。
ウ　すべての藩主が，天皇に領地と領民を返還した。""",
            "ウ→イ→ア"
        ),
        (
            """江戸幕府の北方での対外的な緊張について述べた次の文ア～ウを年代の古い順に正しく並べよ。

ア　レザノフが長崎に来航したが，幕府が冷淡な対応をしたため，ロシア船が樺太や択捉島を攻撃した。
イ　ゴローウニンが国後島に上陸し，幕府の役人に捕らえられ抑留された。
ウ　ラクスマンが根室に来航し，漂流民を届けるとともに通商を求めた。""",
            "ウ→ア→イ"
        ),
        (
            """中居屋重兵衛の生涯の期間におこったできごとについて述べた次のア～ウを，年代の古い順に正しく並べよ。

ア　アヘン戦争がおこり，清がイギリスに敗北した。
イ　異国船打払令が出され，外国船を撃退することが命じられた。
ウ　桜田門外の変がおこり，大老の井伊直弼が暗殺された。""",
            "イ→ア→ウ"
        ),
        (
            """加藤高明が外務大臣として提言を行ってから、内閣総理大臣となり演説を行うまでの時期のできごとについて述べた次のア～ウを，年代の古い順に正しく並べよ。

ア　朝鮮半島において，独立を求める大衆運動である三・一独立運動が展開された。
イ　関東大震災後の混乱のなかで，朝鮮人や中国人に対する殺傷事件がおきた。
ウ　日本政府が，袁世凱政府に対して二十一カ条の要求を突き付けた。""",
            "ウ→ア→イ"
        )
    ]

    # 解答を生成したい新しい問題
    new_question = """9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。"""

    print("解決したい問題:")
    print(new_question)

    # APIキーの取得
    api_key = get_api_key()

    if api_key:
        # Gemini APIを呼び出して問題を解く
        solution = solve_historical_ordering_few_shot(api_key, examples, new_question)

        if solution:
            print(f"\nFew-Shot推論による解答: {solution}")
        else:
            print("\n解答の生成に失敗しました。")
    else:
        print("APIキーが設定されていないため、処理を中止します。")

解決したい問題:
9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。

Geminiに問い合わせ中 (Few-Shot推論)...
  抽出された解答: イ→ウ→ア

Few-Shot推論による解答: イ→ウ→ア
